# VQAデモ（事前学習済みモデル）

このノートブックでは、事前学習済みVQAモデルを使って画像と質問から回答を生成します。

In [ ]:
# 必要なライブラリ
import os
import torch
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib import font_manager
from transformers import ViltProcessor, ViltForQuestionAnswering

# 日本語フォント設定（利用可能なものを自動選択）
def set_japanese_font():
    candidates = [
        "IPAexGothic",
        "IPAGothic",
        "Noto Sans CJK JP",
        "Noto Sans JP",
        "TakaoGothic",
        "Yu Gothic",
        "Hiragino Sans",
    ]
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False

set_japanese_font()

# デバイス
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. モデルの読み込み

VQA用の事前学習済みモデル（VILT）を読み込みます。

In [ ]:
model_id = "dandelin/vilt-b32-finetuned-vqa"
processor = ViltProcessor.from_pretrained(model_id)
model = ViltForQuestionAnswering.from_pretrained(model_id).to(device)
model.eval()

print("model loaded:", model_id)

## 2. 画像と質問で推論

kite.png と re9.png に対して、質問を与えて回答を確認します。

In [ ]:
def vqa_answer(image_path, question):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"{image_path} が見つかりません")

    image = Image.open(image_path).convert("RGB")
    inputs = processor(image, question, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        idx = outputs.logits.argmax(-1).item()
        answer = model.config.id2label[idx]

    return image, answer


tests = [
    ("kite.png", "What is the person doing?"),
    ("kite.png", "What is in the sky?"),
    ("re9.png", "What number is shown?"),
    ("re9.png", "What color is the number?"),
]

for path, q in tests:
    img, ans = vqa_answer(path, q)
    print(f"{path} | Q: {q} | A: {ans}")

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Q: {q}\nA: {ans}")
    plt.show()